In [2]:
# ============================================================
# NDVI OBSERVATIONAL DATA INVENTORY
# scripts/investigate/ndvi.ipynb
# ============================================================

from pathlib import Path
import rasterio
import numpy as np

print("=" * 60)
print("NDVI OBSERVATIONAL DATA INVENTORY")
print("=" * 60)

# ------------------------------------------------------------
# NDVI DIRECTORY (CONFIRMED LOCATION)
# ------------------------------------------------------------
NDVI_DIR = Path(
    r"C:\Projects\Infer RozviDrought\data\real_observations\ndvi"
)

print(f"NDVI directory : {NDVI_DIR}")

if not NDVI_DIR.exists():
    print("Status: DIRECTORY NOT FOUND")
    raise SystemExit

# ------------------------------------------------------------
# Helper: extract YYYYMM from filename
# ------------------------------------------------------------
def extract_yyyymm(path):
    name = path.stem
    for token in name.split("_"):
        if token.isdigit() and len(token) == 6:
            return int(token)
    return None

# ------------------------------------------------------------
# Scan files
# ------------------------------------------------------------
tif_files = sorted(NDVI_DIR.glob("*.tif"))

print("\nNDVI STATUS")
print(f"Total files : {len(tif_files)}")

if len(tif_files) == 0:
    print("No NDVI rasters found.")
    raise SystemExit

months = []

for f in tif_files:
    ym = extract_yyyymm(f)
    if ym:
        months.append(ym)

months = sorted(months)

if months:
    print(f"Months detected : {len(months)}")
    print(f"First month     : {months[0]}")
    print(f"Last month      : {months[-1]}")
else:
    print("No YYYYMM pattern detected in filenames")

# ------------------------------------------------------------
# Inspect sample raster
# ------------------------------------------------------------
sample_file = tif_files[0]

print("\nSample file:")
print(sample_file)

with rasterio.open(sample_file) as src:

    data = src.read(1)

    print("\nRASTER PROPERTIES")
    print(f"CRS       : {src.crs}")
    print(f"Shape     : {data.shape}")
    print(f"Count     : {src.count}")
    print(f"Dtype     : {data.dtype}")
    print(f"Nodata    : {src.nodata}")
    print(f"Bounds    : {src.bounds}")
    print(f"Transform : {src.transform}")

    valid = data[~np.isnan(data)]

    if valid.size > 0:
        print("\nVALUE STATS")
        print(f"Min       : {float(valid.min())}")
        print(f"Max       : {float(valid.max())}")
        print(f"Mean      : {float(valid.mean())}")
    else:
        print("No valid pixels found")

print("\n" + "=" * 60)
print("NDVI INVENTORY COMPLETE")
print("=" * 60)

NDVI OBSERVATIONAL DATA INVENTORY
NDVI directory : C:\Projects\Infer RozviDrought\data\real_observations\ndvi

NDVI STATUS
Total files : 313
Months detected : 313
First month     : 200002
Last month      : 202602

Sample file:
C:\Projects\Infer RozviDrought\data\real_observations\ndvi\ndvi_200002.tif

RASTER PROPERTIES
CRS       : EPSG:4326
Shape     : (3060, 3501)
Count     : 1
Dtype     : float32
Nodata    : None
Bounds    : BoundingBox(left=25.19549793134228, bottom=-22.457882102988037, right=33.05800245559839, top=-15.585770179473698)
Transform : | 0.00, 0.00, 25.20|
| 0.00,-0.00,-15.59|
| 0.00, 0.00, 1.00|

VALUE STATS
Min       : -0.20000000298023224
Max       : 0.9995999932289124
Mean      : 0.543603241443634

NDVI INVENTORY COMPLETE


In [3]:
# ============================================================
# NDVI ↔ ERA5 GRID ALIGNMENT CHECK
# scripts/investigate/ndvi.ipynb
# ============================================================

from pathlib import Path
import rasterio
import numpy as np

print("=" * 60)
print("NDVI ↔ ERA5 GRID ALIGNMENT CHECK")
print("=" * 60)

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

NDVI_DIR = PROJECT_ROOT / "data" / "real_observations" / "ndvi"
ERA5_TEMPLATE = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "t2m" / "t2m_201412.tif"

ndvi_file = NDVI_DIR / "ndvi_200002.tif"

print(f"NDVI file      : {ndvi_file}")
print(f"ERA5 template  : {ERA5_TEMPLATE}")

with rasterio.open(ndvi_file) as ndvi_src:
    ndvi_arr = ndvi_src.read(1).astype(np.float32)

    print("\nNDVI PROPERTIES")
    print(f"CRS       : {ndvi_src.crs}")
    print(f"Shape     : ({ndvi_src.height}, {ndvi_src.width})")
    print(f"Bounds    : {ndvi_src.bounds}")
    print(f"Transform : {ndvi_src.transform}")
    print(f"Res X     : {ndvi_src.transform.a}")
    print(f"Res Y     : {abs(ndvi_src.transform.e)}")
    print(f"Min       : {np.nanmin(ndvi_arr)}")
    print(f"Max       : {np.nanmax(ndvi_arr)}")
    print(f"Mean      : {np.nanmean(ndvi_arr)}")

with rasterio.open(ERA5_TEMPLATE) as era5_src:
    era5_arr = era5_src.read(1).astype(np.float32)

    print("\nERA5 TEMPLATE PROPERTIES")
    print(f"CRS       : {era5_src.crs}")
    print(f"Shape     : ({era5_src.height}, {era5_src.width})")
    print(f"Bounds    : {era5_src.bounds}")
    print(f"Transform : {era5_src.transform}")
    print(f"Res X     : {era5_src.transform.a}")
    print(f"Res Y     : {abs(era5_src.transform.e)}")
    print(f"Min       : {np.nanmin(era5_arr)}")
    print(f"Max       : {np.nanmax(era5_arr)}")
    print(f"Mean      : {np.nanmean(era5_arr)}")

print("\nCOMPARISON")
print(f"Same CRS              : {str(ndvi_src.crs) == str(era5_src.crs)}")
print(f"NDVI finer than ERA5  : {ndvi_src.width > era5_src.width and ndvi_src.height > era5_src.height}")
print("Expected action       : resample NDVI down to ERA5/common inference grid")
print("=" * 60)

NDVI ↔ ERA5 GRID ALIGNMENT CHECK
NDVI file      : C:\Projects\Infer RozviDrought\data\real_observations\ndvi\ndvi_200002.tif
ERA5 template  : C:\Projects\Infer RozviDrought\data\simulated\era5_land\t2m\t2m_201412.tif

NDVI PROPERTIES
CRS       : EPSG:4326
Shape     : (3060, 3501)
Bounds    : BoundingBox(left=25.19549793134228, bottom=-22.457882102988037, right=33.05800245559839, top=-15.585770179473698)
Transform : | 0.00, 0.00, 25.20|
| 0.00,-0.00,-15.59|
| 0.00, 0.00, 1.00|
Res X     : 0.002245788210298804
Res Y     : 0.002245788210298804
Min       : -0.20000000298023224
Max       : 0.9995999932289124
Mean      : 0.543603241443634

ERA5 TEMPLATE PROPERTIES
CRS       : EPSG:4326
Shape     : (70, 80)
Bounds    : BoundingBox(left=25.100114814474466, bottom=-22.500102921341654, right=33.100151408729275, top=-15.500070901368694)
Transform : | 0.10, 0.00, 25.10|
| 0.00,-0.10,-15.50|
| 0.00, 0.00, 1.00|
Res X     : 0.10000045742818513
Res Y     : 0.10000045742818513
Min       : 290.83367919

In [4]:
# ============================================================
# NDVI -> ERA5 GRID RESAMPLING CHECK
# Still investigation only. No proxy/bias modelling yet.
# ============================================================

from pathlib import Path
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

print("=" * 60)
print("NDVI -> ERA5 GRID RESAMPLING CHECK")
print("=" * 60)

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

NDVI_FILE = PROJECT_ROOT / "data" / "real_observations" / "ndvi" / "ndvi_200002.tif"
ERA5_TEMPLATE = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "t2m" / "t2m_201412.tif"

with rasterio.open(NDVI_FILE) as ndvi_src, rasterio.open(ERA5_TEMPLATE) as ref:
    ndvi_arr = ndvi_src.read(1).astype(np.float32)

    # Optional clean-up: keep plausible NDVI range
    ndvi_arr = np.where(
        np.isfinite(ndvi_arr) & (ndvi_arr >= -0.2) & (ndvi_arr <= 1.0),
        ndvi_arr,
        np.nan
    ).astype(np.float32)

    ndvi_resampled = np.full((ref.height, ref.width), np.nan, dtype=np.float32)

    reproject(
        source=ndvi_arr,
        destination=ndvi_resampled,
        src_transform=ndvi_src.transform,
        src_crs=ndvi_src.crs,
        dst_transform=ref.transform,
        dst_crs=ref.crs,
        resampling=Resampling.average,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )

    print(f"Source shape      : {ndvi_arr.shape}")
    print(f"Target shape      : {ndvi_resampled.shape}")
    print(f"Target CRS        : {ref.crs}")
    print(f"Target bounds     : {ref.bounds}")

    print("\nSOURCE NDVI STATS")
    print(f"Min               : {np.nanmin(ndvi_arr)}")
    print(f"Max               : {np.nanmax(ndvi_arr)}")
    print(f"Mean              : {np.nanmean(ndvi_arr)}")

    print("\nRESAMPLED NDVI STATS")
    print(f"Min               : {np.nanmin(ndvi_resampled)}")
    print(f"Max               : {np.nanmax(ndvi_resampled)}")
    print(f"Mean              : {np.nanmean(ndvi_resampled)}")

    valid_pixels = np.isfinite(ndvi_resampled).sum()
    total_pixels = ndvi_resampled.size
    print(f"Valid pixels      : {valid_pixels}/{total_pixels}")

print("=" * 60)
print("RESAMPLING CHECK COMPLETE")
print("=" * 60)

NDVI -> ERA5 GRID RESAMPLING CHECK
Source shape      : (3060, 3501)
Target shape      : (70, 80)
Target CRS        : EPSG:4326
Target bounds     : BoundingBox(left=25.100114814474466, bottom=-22.500102921341654, right=33.100151408729275, top=-15.500070901368694)

SOURCE NDVI STATS
Min               : -0.20000000298023224
Max               : 0.9995999932289124
Mean              : 0.543603241443634

RESAMPLED NDVI STATS
Min               : -0.12096705287694931
Max               : 0.8261405229568481
Mean              : 0.5420119762420654
Valid pixels      : 5598/5600
RESAMPLING CHECK COMPLETE


In [5]:
# ============================================================
# NDVI ↔ CLIMATE TIMELINE OVERLAP CHECK
# Still investigation only. No proxy/bias modelling yet.
# ============================================================

from pathlib import Path

print("=" * 60)
print("NDVI ↔ CLIMATE TIMELINE OVERLAP CHECK")
print("=" * 60)

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

NDVI_DIR = PROJECT_ROOT / "data" / "real_observations" / "ndvi"
ERA5_T2M_DIR = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "t2m"
ERA5_D2M_DIR = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "d2m"
ERA5_SM_DIR = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "sm"
ERA5_PET_DIR = PROJECT_ROOT / "data" / "simulated" / "era5_land" / "pet"

def extract_yyyymm(path):
    stem = path.stem
    for token in stem.split("_"):
        if token.isdigit() and len(token) == 6:
            return token
    return None

def month_set(folder):
    return {
        extract_yyyymm(fp)
        for fp in folder.glob("*.tif")
        if extract_yyyymm(fp) is not None
    }

ndvi_months = month_set(NDVI_DIR)
t2m_months = month_set(ERA5_T2M_DIR)
d2m_months = month_set(ERA5_D2M_DIR)
sm_months = month_set(ERA5_SM_DIR)
pet_months = month_set(ERA5_PET_DIR)

common_climate_months = t2m_months & d2m_months & sm_months & pet_months
ndvi_climate_overlap = ndvi_months & common_climate_months

print(f"NDVI months              : {len(ndvi_months)}")
print(f"Climate-common months    : {len(common_climate_months)}")
print(f"NDVI ∩ climate overlap   : {len(ndvi_climate_overlap)}")

if ndvi_months:
    print(f"NDVI first month         : {min(ndvi_months)}")
    print(f"NDVI last month          : {max(ndvi_months)}")

if common_climate_months:
    print(f"Climate first month      : {min(common_climate_months)}")
    print(f"Climate last month       : {max(common_climate_months)}")

if ndvi_climate_overlap:
    print(f"Overlap first month      : {min(ndvi_climate_overlap)}")
    print(f"Overlap last month       : {max(ndvi_climate_overlap)}")

# timeline gaps relative to 1980-2050 target
target_start = "198001"
target_hist_end = "199912"
target_future_start = "202603"
target_future_end = "205012"

backcast_needed = sorted([m for m in common_climate_months if target_start <= m <= target_hist_end])
future_fill_needed = [m for m in sorted(common_climate_months) if m > max(ndvi_months)] if ndvi_months else []

print("\nGAP SUMMARY")
print(f"Backcast period needed   : {backcast_needed[0] if backcast_needed else None} -> {backcast_needed[-1] if backcast_needed else None}")
print(f"Backcast months count    : {len(backcast_needed)}")

if future_fill_needed:
    print(f"Future fill starts at    : {future_fill_needed[0]}")
    print(f"Future fill ends at      : {future_fill_needed[-1]}")
    print(f"Future fill month count  : {len(future_fill_needed)}")
else:
    print("Future fill month count  : 0")

print("=" * 60)
print("TIMELINE OVERLAP CHECK COMPLETE")
print("=" * 60)

NDVI ↔ CLIMATE TIMELINE OVERLAP CHECK
NDVI months              : 313
Climate-common months    : 523
NDVI ∩ climate overlap   : 301
NDVI first month         : 200002
NDVI last month          : 202602
Climate first month      : 198109
Climate last month       : 202603
Overlap first month      : 200002
Overlap last month       : 202602

GAP SUMMARY
Backcast period needed   : 198109 -> 199912
Backcast months count    : 220
Future fill starts at    : 202603
Future fill ends at      : 202603
Future fill month count  : 1
TIMELINE OVERLAP CHECK COMPLETE
